# Fire & Smoke Detection for BC Wildfires (YOLOv8n fine-tuning)

IAT360 final project. We fine-tune a pretrained **YOLOv8n** on a public Roboflow fire/smoke dataset (classes: `0 = fire`, `1 = smoke`) and evaluate it on our own BC wildfire test set (`bc_test/`, 90 images, 41 of them negatives).

**Notebook layout**
1. Setup (paths)
2. Baseline: train (epochs=10, fraction=0.3), validate on Roboflow valid, evaluate on `bc_test`, label check
3. Tuning: epochs (15 / 20 / 30)
4. Tuning: imgsz (416 / 512 / 704)
5. Tuning: lr0 (0.0005 / 0.003 / 0.006)
6. Combined model (best setting of each experiment)
7. Re-evaluation of every model on the final 90-image test set

**To run it:** download the Roboflow dataset (link in the README / report), then set `DATA` in the setup cell to its `config.yaml`. Run the notebook from the repository root so `bc_test/` and `results/` are found. `bc_test/` contains only labels and `config_bc.yaml` on GitHub (the images are not redistributed, see `bc_test/SOURCES.md`), so put the test images in `bc_test/images/` yourself before evaluating.


In [ ]:
# [SETUP] Paths used by every cell below. Edit DATA only; the other paths are relative to the repository root.
from pathlib import Path
import os

PROJECT = Path.cwd().as_posix()                      # run the notebook from the repository root
DATA = "PATH/TO/including1_2.v1-2026-05-28-10-54am-dataset-l.yolov8/config.yaml"   # <- Roboflow dataset config (EDIT THIS)
TEST = f"{PROJECT}/bc_test/config_bc.yaml"           # our own test set (labels in the repo, images added locally)
RUNS = f"{PROJECT}/runs"                             # training outputs (git-ignored)
RESULTS = f"{PROJECT}/results"                       # summary CSVs and plots that are committed
os.makedirs(RESULTS, exist_ok=True)


## 1. Baseline (epochs=10, fraction=0.3)

In [ ]:
# [BASELINE TRAINING] YOLOv8n fine-tuned on the Roboflow fire/smoke dataset (classes: 0 = fire, 1 = smoke).
# Settings: epochs=10, imgsz=640, batch=16, fraction=0.3 (30% of the training images, (~6.6k images) to balance resource constraints with practical training turnaround times).
# NOTE: optimizer is left at 'auto', so Ultralytics ignored lr0=0.01 and chose AdamW (lr=0.001667, momentum=0.9) - see the training log.
# Output: runs/baseline/ (weights/best.pt, results.png, confusion_matrix.png, ...)
import torch
print(torch.cuda.is_available())

from ultralytics import YOLO
model = YOLO("yolov8n.pt")
model.train(data=DATA,
            epochs=10, imgsz=640, batch=16, lr0=0.01, fraction=0.3,
            project=RUNS, name="baseline")

True
Ultralytics 8.4.204  Python-3.14.2 torch-2.14.1+cu130 CUDA:0 (NVIDIA GeForce GTX 1660 Ti, 6144MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:/iat360/including1_2.v1-2026-05-28-10-54am-dataset-l.yolov8/config.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=0.3, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, n

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x000002C487DE1EF0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.0

### Baseline on the Roboflow validation split

In [ ]:
# [BASELINE EVAL 1/3] Validate the baseline best.pt on the Roboflow validation split (6,652 images, 13,268 boxes).
from ultralytics import YOLO
model = YOLO(f"{RUNS}/baseline/weights/best.pt")
m = model.val(data=DATA)

Ultralytics 8.4.204  Python-3.14.2 torch-2.14.1+cu130 CUDA:0 (NVIDIA GeForce GTX 1660 Ti, 6144MiB)
Model summary (fused): 73 layers, 3,132,950 parameters, 0 gradients, 8.6 GFLOPs
val: Fast image access  (ping: 0.1±0.0 ms, read: 164.1±68.3 MB/s, size: 145.3 KB)
val: Scanning D:\iat360\including1_2.v1-2026-05-28-10-54am-dataset-l.yolov8\valid\labels.cache... 6652 images, 355 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 6652/6652 1.9Git/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 416/416 4.2it/s 1:390.2sss
                   all       6652      13268      0.756      0.704      0.777      0.469
                  fire       4641       7097      0.757      0.731      0.797      0.489
                 smoke       5056       6171      0.755      0.676      0.758      0.449
Speed: 2.2ms preprocess, 4.8ms inference, 0.0ms loss, 1.2ms postprocess per image
Results saved to D:\iat360\Computer-Vision-Project\runs\detect\val


### Baseline on our own test set (bc_test)

In [ ]:
# [BASELINE EVAL 2/3] Evaluate the baseline on our own test set (bc_test, 90 images) using bc_test/config_bc.yaml.
# Earlier runs of this cell (saved as runs/baseline_bc90*) gave near-zero mAP because some label files had class IDs 0/1 swapped.
# This output is the re-run after the labels were fixed.
from ultralytics import YOLO
model = YOLO(f"{RUNS}/baseline/weights/best.pt")
m = model.val(data=TEST,
              project=RUNS, name="baseline_bc90")

Ultralytics 8.4.204  Python-3.14.2 torch-2.14.1+cu130 CUDA:0 (NVIDIA GeForce GTX 1660 Ti, 6144MiB)
Model summary (fused): 73 layers, 3,132,950 parameters, 0 gradients, 8.6 GFLOPs
val: Fast image access  (ping: 0.1±0.0 ms, read: 899.9±708.3 MB/s, size: 235.7 KB)
val: Scanning D:\iat360\Computer-Vision-Project\bc_test\labels... 89 images, 41 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 89/89 1.1Kit/s 0.1s
val: New cache created: D:\iat360\Computer-Vision-Project\bc_test\labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 6/6 1.1it/s 5.3s0.3s4
                   all         89        144      0.353       0.41       0.24        0.1
                  fire         44        110      0.414      0.291      0.227     0.0799
                 smoke         29         34      0.291      0.529      0.253      0.121
Speed: 1.0ms preprocess, 5.4ms inference, 0.0ms loss, 1.9ms postprocess per image
Results saved to D:\iat360\Computer-Vi

In [ ]:
# [LABEL DIAGNOSTIC] Read-only check, it does not modify any file.
# For every labelled test image, count how many ground-truth boxes agree with the model's predictions (IoU >= 0.3)
# with the class IDs as they are vs. with 0/1 swapped. verdict: ok / FLIP? / unclear.
# Used to find annotation files whose class order was reversed.
from ultralytics import YOLO
from pathlib import Path
import numpy as np

model = YOLO(f"{RUNS}/baseline/weights/best.pt")
root = Path(f"{PROJECT}/bc_test")

def xyxy(b):
    x, y, w, h = b
    return [x - w/2, y - h/2, x + w/2, y + h/2]

def iou(a, b):
    a, b = xyxy(a), xyxy(b)
    ix = max(0, min(a[2], b[2]) - max(a[0], b[0]))
    iy = max(0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ix * iy
    u = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter
    return inter / u if u > 0 else 0

print("file\tgt_boxes\tmatch_orig\tmatch_swapped\tverdict")
for p in sorted((root/"images").glob("*.*")):
    lab = root/"labels"/(p.stem + ".txt")
    gt = []
    if lab.exists():
        for ln in lab.read_text().splitlines():
            q = ln.split()
            if len(q) == 5:
                gt.append((int(q[0]), list(map(float, q[1:]))))
    if not gt:
        continue
    r = model.predict(p, conf=0.25, verbose=False)[0]
    pred = list(zip(r.boxes.cls.int().tolist(), r.boxes.xywhn.tolist()))

    def score(swap):
        s = 0
        for c, b in gt:
            c2 = 1 - c if swap else c
            if any(pc == c2 and iou(b, pb) >= 0.3 for pc, pb in pred):
                s += 1
        return s

    so, ss = score(False), score(True)
    v = "FLIP?" if ss > so else ("ok" if so > ss else "unclear")
    print(p.name, len(gt), so, ss, v, sep="\t")

file	gt_boxes	match_orig	match_swapped	verdict
dy_img01.jpg	9	2	0	ok
dy_img02.jpg	3	1	0	ok
dy_img03.jpg	5	3	0	ok
dy_img04.jpg	4	3	0	ok
dy_img05.jpg	5	3	0	ok
dy_img06.jpg	5	2	0	ok
dy_img07.jpg	2	1	0	ok
dy_img08.jpg	3	2	0	ok
dy_img09.jpg	3	2	0	ok
dy_img10.jpg	2	1	0	ok
dy_img12.jpg	3	1	0	ok
dy_img13.jpg	1	1	0	ok
dy_img14.jpg	4	1	0	ok
dy_img15.jpg	2	0	0	unclear
dy_img16.jpg	1	1	0	ok
dy_img17.jpg	1	1	0	ok
dy_img18.jpg	6	2	0	ok
dy_img19.jpg	3	2	0	ok
dy_img20.jpg	2	2	0	ok
dy_img21.jpg	4	4	0	ok
dy_img22.jpg	1	1	0	ok
dy_img23.jpg	1	1	0	ok
dy_img24.jpg	2	1	0	ok
img10.jpg	3	1	0	ok
img11-wildfire.JPG	4	2	0	ok
img11.jpg	1	1	0	ok
img12-wildfire.JPG	8	1	0	ok
img12.jpg	3	2	0	ok
img13-wildfire.JPG	6	2	0	ok
img14-wildfire.JPG	4	0	0	unclear
img15-greasefire.JPG	1	1	0	ok
img16-carfire.JPG	2	1	0	ok
img17-fire.JPG	2	1	0	ok
img18-fire.JPG	2	1	0	ok
img20.jpg	1	1	0	ok
img21.jpg	3	2	0	ok
img22.jpg	5	3	1	ok
img23.jpg	4	1	0	ok
img24.jpg	2	2	0	ok
img25.jpg	2	1	0	ok
img26.jpg	3	1	0	ok
img3-wildfire.JPG	3	0	0	unclea

In [ ]:
# [BASELINE EVAL 3/3 - FINAL] Delete the stale label cache, then evaluate the baseline on bc_test again.
# Prints P, R, F1, mAP50, mAP50-95 (F1 = 2PR / (P + R)). Results saved to runs/baseline_bc_final/.
from pathlib import Path
from ultralytics import YOLO

cache = Path(f"{PROJECT}/bc_test/labels.cache")
if cache.exists():
    cache.unlink()

model = YOLO(f"{RUNS}/baseline/weights/best.pt")
m = model.val(data=TEST,
              project=RUNS, name="baseline_bc_final")

P, R = m.box.mp, m.box.mr
print("P", round(P, 3), "R", round(R, 3), "F1", round(2*P*R/(P+R), 3),
      "mAP50", round(m.box.map50, 3), "mAP50-95", round(m.box.map, 3))

Ultralytics 8.4.204  Python-3.14.2 torch-2.14.1+cu130 CUDA:0 (NVIDIA GeForce GTX 1660 Ti, 6144MiB)
Model summary (fused): 73 layers, 3,132,950 parameters, 0 gradients, 8.6 GFLOPs
val: Fast image access  (ping: 0.0±0.0 ms, read: 523.3±380.4 MB/s, size: 74.5 KB)
val: Scanning D:\iat360\Computer-Vision-Project\bc_test\labels... 89 images, 41 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 89/89 1.8Kit/s 0.0s
val: New cache created: D:\iat360\Computer-Vision-Project\bc_test\labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 6/6 1.1it/s 5.6s0.3s8
                   all         89        144      0.353       0.41       0.24        0.1
                  fire         44        110      0.414      0.291      0.227     0.0799
                 smoke         29         34      0.291      0.529      0.253      0.121
Speed: 2.1ms preprocess, 5.1ms inference, 0.0ms loss, 1.5ms postprocess per image
Results saved to D:\iat360\Computer-Vis

## 2. Tuning: epochs (15 / 20 / 30)
Only `epochs` changes; everything else is the same as the baseline.

In [ ]:
# [TUNING: epochs = 15] Same settings as the baseline; the only change is epochs (10 -> 15).
# Step 1: train on the Roboflow data (fraction=0.3). Output: runs/epochs15/
# Step 2: evaluate best.pt on our test set (bc_test). Output: runs/epochs15_bc/, printed as the TEST line below.
# NOTE: lr0 is ignored here too (optimizer='auto' -> AdamW, lr=0.001667), same as the baseline.
from ultralytics import YOLO

E = 15
model = YOLO("yolov8n.pt")
model.train(data=DATA,
            epochs=E, imgsz=640, batch=16, lr0=0.01, fraction=0.3,
            project=RUNS, name=f"epochs{E}")

best = YOLO(f"{RUNS}/epochs{E}/weights/best.pt")
m = best.val(data=TEST,
             project=RUNS, name=f"epochs{E}_bc")
P, R = m.box.mp, m.box.mr
print("TEST  P", round(P,3), "R", round(R,3), "F1", round(2*P*R/(P+R),3),
      "mAP50", round(m.box.map50,3), "mAP50-95", round(m.box.map,3))

New https://pypi.org/project/ultralytics/8.4.205 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.204  Python-3.14.2 torch-2.14.1+cu130 CUDA:0 (NVIDIA GeForce GTX 1660 Ti, 6144MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:/iat360/including1_2.v1-2026-05-28-10-54am-dataset-l.yolov8/config.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=15, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=0.3, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_

In [ ]:
# [TUNING: epochs = 20] Same settings as the baseline; the only change is epochs (10 -> 20).
from ultralytics import YOLO

E = 20
model = YOLO("yolov8n.pt")
model.train(data=DATA,
            epochs=E, imgsz=640, batch=16, lr0=0.01, fraction=0.3,
            project=RUNS, name=f"epochs{E}")

best = YOLO(f"{RUNS}/epochs{E}/weights/best.pt")
m = best.val(data=TEST,
             project=RUNS, name=f"epochs{E}_bc")
P, R = m.box.mp, m.box.mr
print("TEST  P", round(P,3), "R", round(R,3), "F1", round(2*P*R/(P+R),3),
      "mAP50", round(m.box.map50,3), "mAP50-95", round(m.box.map,3))

New https://pypi.org/project/ultralytics/8.4.205 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.204  Python-3.14.2 torch-2.14.1+cu130 CUDA:0 (NVIDIA GeForce GTX 1660 Ti, 6144MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:/iat360/including1_2.v1-2026-05-28-10-54am-dataset-l.yolov8/config.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=0.3, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_

In [ ]:
# [TUNING: epochs = 30] Same settings as the baseline; the only change is epochs (10 -> 30).
from ultralytics import YOLO

E = 30
model = YOLO("yolov8n.pt")
model.train(data=DATA,
            epochs=E, imgsz=640, batch=16, lr0=0.01, fraction=0.3,
            project=RUNS, name=f"epochs{E}")

best = YOLO(f"{RUNS}/epochs{E}/weights/best.pt")
m = best.val(data=TEST,
             project=RUNS, name=f"epochs{E}_bc")
P, R = m.box.mp, m.box.mr
print("TEST  P", round(P,3), "R", round(R,3), "F1", round(2*P*R/(P+R),3),
      "mAP50", round(m.box.map50,3), "mAP50-95", round(m.box.map,3))

New https://pypi.org/project/ultralytics/8.4.206 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.204  Python-3.14.2 torch-2.14.1+cu130 CUDA:0 (NVIDIA GeForce GTX 1660 Ti, 6144MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:/iat360/including1_2.v1-2026-05-28-10-54am-dataset-l.yolov8/config.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=0.3, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_

## 3. Tuning: imgsz (416 / 512 / 704)
Only `imgsz` changes; everything else is the same as the baseline.

In [14]:
# [TUNING: imgsz = 416 / 512 / 704] One change vs. the baseline per run (baseline: AdamW lr=0.001667, epochs=10, imgsz=640, batch=16, fraction=0.3).
# Each run: train -> validate on the Roboflow valid split -> evaluate on our test set (bc_test). Summary saved to results/tuning_imgsz.csv.
import csv, gc, os, torch
from ultralytics import YOLO

base = dict(epochs=10, imgsz=640, batch=16, fraction=0.3)

def run_jobs(jobs, out_csv):
    rows = []
    for name, change in jobs:
        try:
            best_path = f"{RUNS}/{name}/weights/best.pt"
            if not os.path.exists(best_path):          # skip training if this run already finished
                YOLO("yolov8n.pt").train(data=DATA, project=RUNS, name=name, **{**base, **change})
            best = YOLO(best_path)
            mv = best.val(data=DATA, project=RUNS, name=f"{name}_valid")
            mt = best.val(data=TEST, project=RUNS, name=f"{name}_bc")
            row = {"run": name}
            for tag, m in (("valid", mv), ("test", mt)):
                P, R = m.box.mp, m.box.mr
                row[f"{tag}_P"] = round(P, 3); row[f"{tag}_R"] = round(R, 3)
                row[f"{tag}_F1"] = round(2*P*R/(P+R), 3) if P + R else 0
                row[f"{tag}_mAP50"] = round(m.box.map50, 3); row[f"{tag}_mAP50-95"] = round(m.box.map, 3)
            rows.append(row); print(row)
            with open(out_csv, "w", newline="") as f:
                w = csv.DictWriter(f, fieldnames=list(rows[0].keys())); w.writeheader(); w.writerows(rows)
        except Exception as e:
            print("FAILED", name, e)
        gc.collect(); torch.cuda.empty_cache()

run_jobs([(f"imgsz_{v}", dict(imgsz=v)) for v in (416, 512, 704)],
         f"{PROJECT}/results/tuning_imgsz.csv")

New https://pypi.org/project/ultralytics/8.4.208 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.204  Python-3.14.2 torch-2.14.1+cu130 CUDA:0 (NVIDIA GeForce GTX 1660 Ti, 6144MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:/iat360/including1_2.v1-2026-05-28-10-54am-dataset-l.yolov8/config.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=0.3, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=416, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_

## 4. Tuning: lr0 (0.0005 / 0.003 / 0.006)
Only `lr0` changes; everything else is the same as the baseline.

In [15]:
# [TUNING: lr0 = 0.0005 / 0.003 / 0.006] optimizer is set to AdamW explicitly (momentum 0.9) because optimizer='auto' ignores lr0.
# The baseline's automatic choice was AdamW with lr=0.001667, so these runs differ from the baseline only in the learning rate.
# Needs run_jobs() and the paths from the imgsz cell above. Summary saved to results/tuning_lr0.csv.
run_jobs([(f"lr0_{v}", dict(optimizer="AdamW", momentum=0.9, lr0=v)) for v in (0.0005, 0.003, 0.006)],
         f"{PROJECT}/results/tuning_lr0.csv")

New https://pypi.org/project/ultralytics/8.4.208 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.204  Python-3.14.2 torch-2.14.1+cu130 CUDA:0 (NVIDIA GeForce GTX 1660 Ti, 6144MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:/iat360/including1_2.v1-2026-05-28-10-54am-dataset-l.yolov8/config.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=0.3, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.0005, lrf=0.01, mask_ratio=4, ma

In [16]:
# [TUNING retry: imgsz = 704] The first attempt crashed in a DataLoader worker during epoch 2.
# The partial run is deleted so it is retrained from scratch. workers=2 only lowers the data-loading load, not the model settings.
# Summary saved to results/tuning_imgsz_704.csv.
import shutil
shutil.rmtree(f"{RUNS}/imgsz_704", ignore_errors=True)
run_jobs([("imgsz_704", dict(imgsz=704, workers=2))],
         f"{PROJECT}/results/tuning_imgsz_704.csv")

New https://pypi.org/project/ultralytics/8.4.208 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.204  Python-3.14.2 torch-2.14.1+cu130 CUDA:0 (NVIDIA GeForce GTX 1660 Ti, 6144MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:/iat360/including1_2.v1-2026-05-28-10-54am-dataset-l.yolov8/config.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=0.3, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=704, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_

## 5. Combined model
Best setting of each experiment, trained together.

In [17]:
# [COMBINED MODEL] Best setting of each experiment, set explicitly:
# epochs=30 (highest mAP50-95 among the epoch runs), imgsz=512 (same valid mAP50 as 704 but about 2x faster and lighter),
# AdamW lr0=0.0005 (best valid mAP50 among the lr runs). momentum 0.9 as in the baseline.
# Summary saved to results/tuning_combined.csv.
run_jobs([("combined_e30_sz512_lr0.0005",
           dict(epochs=30, imgsz=512, optimizer="AdamW", momentum=0.9, lr0=0.0005, workers=2))],
         f"{PROJECT}/results/tuning_combined.csv")

New https://pypi.org/project/ultralytics/8.4.208 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.204  Python-3.14.2 torch-2.14.1+cu130 CUDA:0 (NVIDIA GeForce GTX 1660 Ti, 6144MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:/iat360/including1_2.v1-2026-05-28-10-54am-dataset-l.yolov8/config.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=0.3, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=512, iou=0.7, kobj=1.0, line_width=None, lr0=0.0005, lrf=0.01, mask_ratio=4, ma

## 6. Re-evaluation on the final 90-image test set
No retraining; each saved `best.pt` is evaluated again.

In [18]:
# [RE-EVALUATION ON THE 90-IMAGE TEST SET] No retraining: every saved best.pt is evaluated again on bc_test (added one missing image).
# Only the baseline and the combined model also save plots (confusion matrix, prediction images). Summary: results/test_eval_90.csv
import csv, os
from ultralytics import YOLO

cache = f"{PROJECT}/bc_test/labels.cache"
if os.path.exists(cache):
    os.remove(cache)                       # force Ultralytics to re-read the labels

models = ["baseline", "epochs15", "epochs20", "epochs30", "imgsz_416", "imgsz_512", "imgsz_704",
          "lr0_0.0005", "lr0_0.003", "lr0_0.006", "combined_e30_sz512_lr0.0005"]
rows = []
for name in models:
    best = YOLO(f"{RUNS}/{name}/weights/best.pt")
    m = best.val(data=TEST, project=RUNS, name=f"{name}_bc90",
                 plots=name in (models[0], models[-1]), verbose=False)
    P, R = float(m.box.mp), float(m.box.mr)
    rows.append({"run": name, "P": round(P, 3), "R": round(R, 3),
                 "F1": round(2*P*R/(P+R), 3) if P + R else 0,
                 "mAP50": round(float(m.box.map50), 3), "mAP50-95": round(float(m.box.map), 3),
                 "fire_mAP50": round(float(m.box.ap50[0]), 3), "smoke_mAP50": round(float(m.box.ap50[1]), 3),
                 "fire_R": round(float(m.box.r[0]), 3), "smoke_R": round(float(m.box.r[1]), 3)})
    print(rows[-1])

with open(f"{PROJECT}/results/test_eval_90.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0].keys())); w.writeheader(); w.writerows(rows)

Ultralytics 8.4.204  Python-3.14.2 torch-2.14.1+cu130 CUDA:0 (NVIDIA GeForce GTX 1660 Ti, 6144MiB)
Model summary (fused): 73 layers, 3,132,950 parameters, 0 gradients, 8.6 GFLOPs
val: Fast image access  (ping: 0.0±0.0 ms, read: 570.2±370.8 MB/s, size: 132.3 KB)
val: Scanning D:\iat360\Computer-Vision-Project\bc_test\labels... 90 images, 41 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 90/90 905.5it/s 0.1s
val: New cache created: D:\iat360\Computer-Vision-Project\bc_test\labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 6/6 1.1s/it 6.8s0.3s3
                   all         90        148      0.323      0.407      0.233     0.0959
Speed: 2.9ms preprocess, 6.8ms inference, 0.0ms loss, 2.4ms postprocess per image
Results saved to D:\iat360\Computer-Vision-Project\runs\baseline_bc90-6
{'run': 'baseline', 'P': 0.323, 'R': 0.407, 'F1': 0.36, 'mAP50': 0.233, 'mAP50-95': 0.096, 'fire_mAP50': 0.216, 'smoke_mAP50': 0.25, 'fire_R': 